# Krishi Drishti - P08: Two-Stage Transfer Learning with MobileNetV2
Trains MobileNetV2 on 13 Tomato & Potato leaf disease classes.
- Stage 1: Freeze base model, train head (Adam lr=1e-3).
- Stage 2: Fine-tune top 30 layers with BatchNorm frozen (Adam lr=1e-5).
- Input: 224x224x3 with embedded MobileNetV2 [-1, 1] normalization layer.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

split_df = pd.read_csv("ml-training/data/split.csv")
with open("ml-training/data/class_names.json") as f:
    class_names = json.load(f)

class_to_idx = {name: idx for idx, name in enumerate(class_names)}
num_classes = len(class_names)
print(f"Num classes: {num_classes}")


### Build tf.data pipeline with embedded preprocessing


In [ ]:
def parse_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (224, 224))
    return img, label

def get_dataset(subset_df, batch_size=64, is_train=False):
    paths = subset_df["path"].values
    labels = [class_to_idx[l] for l in subset_df["label"].values]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    if is_train:
        ds = ds.shuffle(1024, seed=42)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = get_dataset(split_df[split_df["split"] == "train"], is_train=True)
val_ds = get_dataset(split_df[split_df["split"] == "val"], is_train=False)


### Build MobileNetV2 Model with Preprocessing Inside


In [ ]:
inputs = tf.keras.Input(shape=(224, 224, 3), name="input_image")
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="MobileNetV2_LeafDisease")
model.summary()


### Stage 1: Train Head


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
history_s1 = model.fit(train_ds, validation_data=val_ds, epochs=3)


### Stage 2: Fine-Tuning Top 30 Layers (BatchNorm Frozen)


In [ ]:
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30

for i, layer in enumerate(base_model.layers):
    if i < fine_tune_at or isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("ml-training/models/best.keras", monitor="val_accuracy", save_best_only=True)
]

history_s2 = model.fit(train_ds, validation_data=val_ds, epochs=3, callbacks=callbacks)
model.save("ml-training/models/best.keras")
print("Saved best.keras successfully!")
